# Visualisierung der Kriminalitätsraten
Vergleich der Überrepräsentation (Deutsch vs. Nichtdeutsch) für ausgewählte Delikte.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Daten laden
df = pd.read_csv('../data/03_processed/master_kriminalitaet_1984_2024.csv', sep=';')
df_plot = df[df['nationalitaet'].isin(['Deutsch', 'Nichtdeutsch'])].copy()

# Drei exemplarische Delikte für den Vergleich
delikte = [
    'Diebstahl und Unterschlagung',
    'Gefährliche und schwere Körperverletzung',
    'Straftaten nach dem Betäubungsmittelgesetz'
]

df_subset = df_plot[df_plot['straftat_bezeichnung'].isin(delikte)]

# Pivotieren: pro (Jahr, Delikt) die Raten für Deutsch/Nichtdeutsch nebeneinander
df_pivot = df_subset.pivot_table(
    index=['jahr', 'straftat_bezeichnung'],
    columns='nationalitaet',
    values=['verurteilte_pro_100k', 'tatverdaechtige_pro_100k']
)

# Überrepräsentationsfaktor = Rate(Nichtdeutsch) / Rate(Deutsch)
df_pivot['faktor_justiz'] = df_pivot[('verurteilte_pro_100k', 'Nichtdeutsch')] / df_pivot[('verurteilte_pro_100k', 'Deutsch')]
df_pivot['faktor_polizei'] = df_pivot[('tatverdaechtige_pro_100k', 'Nichtdeutsch')] / df_pivot[('tatverdaechtige_pro_100k', 'Deutsch')]

df_factors = df_pivot[['faktor_justiz', 'faktor_polizei']].copy()
df_factors.columns = ['faktor_justiz', 'faktor_polizei']
df_factors = df_factors.reset_index()

# Inf-Werte bereinigen (z.B. Division durch 0 in Jahren ohne PKS-Daten 1984-1986)
df_factors.replace([np.inf, -np.inf], np.nan, inplace=True)

# Plots: Überrepräsentationsfaktor pro Delikt über die Jahre
sns.set_theme(style="whitegrid")

for delikt in delikte:
    plt.figure(figsize=(10, 5))
    df_delikt = df_factors[df_factors['straftat_bezeichnung'] == delikt]
    
    ax = sns.lineplot(data=df_delikt, x='jahr', y='faktor_polizei', 
                      linewidth=2, marker='o', color='red', label='Polizei (Tatverdächtige)')
    
    sns.lineplot(data=df_delikt, x='jahr', y='faktor_justiz', 
                 linewidth=2, marker='o', color='blue', label='Justiz (Verurteilte)', ax=ax)
    
    ax.axhline(1.0, ls='--', color='black', alpha=0.7, label='Keine Überrepräsentation (Faktor 1.0)')
    
    ax.set_title(f'Überrepräsentationsfaktor: {delikt}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Faktor (Nichtdeutsch / Deutsch)', fontsize=11)
    ax.set_xlabel('Jahr', fontsize=11)
    ax.legend(title='Quelle')
    
    plt.tight_layout()
    plt.show()